# Evaluation

This notebook evaluates an EN/FR semantic-search index (multilingual-E5 + FAISS).

**What changed in this version, and why.** The previous version reported four
numbers — average similarity, EN/FR score gap, cross-lingual overlap, and score
distribution. None of those measure whether the *correct* document is actually
retrieved: a model can return a high cosine score for the **wrong** document, so
"average similarity = 0.86" does not prove relevance.

Following two standard references on search evaluation —
Shakkari, *Ranking Metrics for Evaluating QA Systems* (Top-N / Precision@k / MRR /
MAP / nDCG), and Zheng et al., *Semantic Search Evaluation* (arXiv:2410.21549,
which confirms MAP and nDCG as the standard relevance metrics and adds an
LLM-judged "On-Topic Rate") — we restructure the notebook into:

1. **Primary — ground-truth ranking metrics:** Hit Rate@k (Top-N accuracy),
   MRR, and nDCG@k. Every test query is built from a known record, so that
   record is the *gold answer*; we measure whether (and at what rank) it comes
   back. This is the part the old notebook was missing.
2. **Secondary — score-health diagnostics:** the original four metrics, kept but
   reframed as *diagnostics* (they describe score behaviour, not correctness).
3. **Optional — On-Topic Rate (LLM judge):** a scaffold for the arXiv approach,
   for queries where no single record is the "right" answer.

Pipeline: load embeddings -> build FAISS -> generate labelled queries ->
retrieve top-K -> compute ranking metrics + diagnostics -> save CSV/plots.

In [ ]:
import pandas as pd
import numpy as np
import faiss
import torch
from sentence_transformers import SentenceTransformer
import random
import math
import matplotlib.pyplot as plt

In [ ]:
data_folder = "D:/Harshita Ajmani/Code_harshu/NLP/data/"
model_name  = "intfloat/multilingual-e5-base"

# k values for the ranking metrics. We retrieve up to EVAL_K and slice.
K_VALUES = [1, 3, 5, 10]
EVAL_K   = max(K_VALUES)
TOP_K    = 5            # kept for the diagnostic/consistency sections

MIN_SCORE = 0.7         # diagnostic threshold only (E5 cosine sims run high)
N_PER_TYPE = 25         # queries per query-type (raise for tighter estimates)

random.seed(42)
np.random.seed(42)

In [ ]:
# Loading embeddings and building the FAISS indexes
#
# IMPORTANT: row i of embeddings_en / embeddings_fr must correspond to row i of
# index_data.csv. The ground-truth metrics below rely on this alignment
# (FAISS returns positional ids, which we compare against each record's row id).

embeddings_en = np.load(f"{data_folder}/embeddings_en.npy").astype('float32')
embeddings_fr = np.load(f"{data_folder}/embeddings_fr.npy").astype('float32')

dataframe = pd.read_csv(f"{data_folder}/index_data.csv")

assert len(dataframe) == len(embeddings_en) == len(embeddings_fr), \
    "Row count mismatch: embeddings are not aligned with index_data.csv"

device = "cuda" if torch.cuda.is_available() else "cpu"
model  = SentenceTransformer(model_name, device=device)

dim      = embeddings_en.shape[1]
index_en = faiss.IndexFlatIP(dim)
index_fr = faiss.IndexFlatIP(dim)
index_en.add(embeddings_en)
index_fr.add(embeddings_fr)

print(f"{len(dataframe):,} records indexed | dim={dim}")

In [ ]:
# Search function (defined early so the sanity check below can use it)
def search(query, search_idx, top_k=EVAL_K):
    """Encode a query and return (indices, scores) from the chosen index."""
    vec = model.encode(f"query: {query}", normalize_embeddings=True
                       ).astype('float32').reshape(1, -1)
    index = index_en if search_idx == 'en' else index_fr
    scores, indices = index.search(vec, top_k)
    return indices[0].tolist(), scores[0].tolist()

In [ ]:
# Sanity check: self-retrieval.
# Encode a record's OWN title (passage side) and confirm it comes back at rank 1
# in its own index. If this fails, embeddings are misaligned with the CSV and the
# ground-truth metrics below will be meaningless.
def passage_search(text, search_idx, top_k=1):
    vec = model.encode(f"passage: {text}", normalize_embeddings=True
                      ).astype('float32').reshape(1, -1)
    index = index_en if search_idx == 'en' else index_fr
    _, indices = index.search(vec, top_k)
    return indices[0].tolist()

probe = dataframe.sample(5, random_state=0)
self_hits = sum(i == passage_search(dataframe.loc[i, 'title_en'], 'en')[0]
                for i in probe.index)
print(f"Self-retrieval sanity: {self_hits}/5 records found themselves at rank 1")
if self_hits < 4:
    print("  WARNING: low self-retrieval -> check embedding/CSV alignment "
          "or passage-encoding prefix.")

In [ ]:
# Data-quality filter: keep records with 5+ word titles in BOTH languages and
# where EN/FR titles differ (avoids trivial / duplicate pairs).
filtered_records = dataframe[
    (dataframe['title_en'].str.split().str.len() >= 5) &
    (dataframe['title_fr'].str.split().str.len() >= 5) &
    (dataframe['title_en'] != dataframe['title_fr'])
].copy().reset_index()          # 'index' column = original row id == FAISS id
print(f"{len(filtered_records):,} records pass quality filter")

# Build search queries from titles by dropping stop words / very short tokens.
wr_eng = {'the','a','an','of','to','and','in','for','with','by','on','at',
          'from','is','are','was','its','this','that'}
wr_fr  = {'le','la','les','de','du','des','et','en','un','une','au','aux',
          'par','sur','pour','dans','ce','qui','que'}

def extract_words(title, lang, n=6):
    """Return up to n meaningful consecutive words, skipping stop/short words."""
    stops = wr_eng if lang == 'en' else wr_fr
    words = [w for w in title.split() if w.lower() not in stops and len(w) > 3]
    if len(words) < 2:
        return None
    return " ".join(words[:n])

In [ ]:
# Generate labelled test queries.
#  - 25 monolingual EN (en->en)        - 25 cross-lingual EN->FR (en->fr)
#  - 25 monolingual FR (fr->fr)        - 25 cross-lingual FR->EN (fr->en)
# Each query carries true_idx = the source record's row id = the GOLD answer.
#
# Note: en->en and en->fr share the same random_state, so they sample the same
# records; likewise fr->fr and fr->en. That is deliberate -- it lets the
# cross-lingual consistency check (below) match EN and FR queries by true_idx.
print("Generating queries...")

def build_queries(records, query_lang, search_lang, n):
    sampled = records.sample(n=n, random_state=42)
    out = []
    for _, row in sampled.iterrows():
        q = extract_words(row[f'title_{query_lang}'], query_lang)
        if not q:
            continue
        out.append({
            "query":      q,
            "query_lang": query_lang,
            "search_idx": search_lang,
            "query_type": f"{query_lang}->{search_lang}",
            "true_idx":   row['index'],      # GOLD row id (aligns with FAISS)
            "title_en":   row['title_en'],
            "title_fr":   row['title_fr'],
        })
    return out

all_queries = (
    build_queries(filtered_records, 'en', 'en', N_PER_TYPE) +
    build_queries(filtered_records, 'fr', 'fr', N_PER_TYPE) +
    build_queries(filtered_records, 'en', 'fr', N_PER_TYPE) +
    build_queries(filtered_records, 'fr', 'en', N_PER_TYPE)
)
print(f"Generated {len(all_queries)} queries")

In [ ]:
# Run every query, retrieve EVAL_K results, and record WHERE the gold record
# landed (gold_rank: 1-based rank, or None if it never appeared in top EVAL_K).
print(f"Running {len(all_queries)} queries (top-{EVAL_K})...")

results = []
for i, q in enumerate(all_queries):
    if (i + 1) % 25 == 0:
        print(f"   {i+1}/{len(all_queries)} done...")

    top_indices, top_scores = search(q['query'], q['search_idx'], EVAL_K)

    gold_rank = next((r + 1 for r, idx in enumerate(top_indices)
                      if idx == q['true_idx']), None)

    results.append({
        "query":       q['query'],
        "query_type":  q['query_type'],
        "search_idx":  q['search_idx'],
        "true_idx":    q['true_idx'],
        "gold_rank":   gold_rank,            # <-- enables the ranking metrics
        "top_score":   round(top_scores[0], 4),
        "worst_score": round(top_scores[-1], 4),
        "gap":         round(top_scores[0] - top_scores[-1], 4),
        "all_scores":  top_scores,
        "top_indices": top_indices,
    })

results_df = pd.DataFrame(results)
found = results_df['gold_rank'].notna().sum()
print(f"Gold record retrieved within top-{EVAL_K} for {found}/{len(results_df)} queries")

## Primary metrics — ground-truth ranking quality

These are the metrics the two references recommend. Because each query has
exactly **one** gold record, the definitions simplify:

- **Hit Rate@k** (= Top-N accuracy = Recall@k here): fraction of queries whose
  gold record appears in the top *k*. "Does the right answer show up at all?"
- **MRR** (Mean Reciprocal Rank): mean of `1/rank` of the gold record (0 if not
  found). Rewards putting the right answer *high*.
- **nDCG@k**: with one binary-relevant doc, `nDCG@k = 1/log2(rank+1)` if the gold
  is within *k*, else 0 — a rank-discounted version of Hit Rate.
- For this single-gold setup, **Precision@k = Hit Rate@k / k** and
  **MAP = MRR**, so we don't print them separately (they add no information).

In [ ]:
# Ranking-metric implementations (single gold record per query)
def hit_rate_at_k(ranks, k):
    return np.mean([(r is not None and r <= k) for r in ranks])

def mrr(ranks):
    return np.mean([(1.0 / r) if r is not None else 0.0 for r in ranks])

def ndcg_at_k(ranks, k):
    # IDCG = 1/log2(1+1) = 1 for a single relevant doc, so nDCG = DCG.
    return np.mean([(1.0 / math.log2(r + 1)) if (r is not None and r <= k) else 0.0
                    for r in ranks])

query_types = ['en->en', 'fr->fr', 'en->fr', 'fr->en']

def metrics_for(df):
    ranks = df['gold_rank'].tolist()
    row = {f"Hit@{k}": hit_rate_at_k(ranks, k) for k in K_VALUES}
    row["MRR"] = mrr(ranks)
    for k in K_VALUES:
        row[f"nDCG@{k}"] = ndcg_at_k(ranks, k)
    return row

summary = {"OVERALL": metrics_for(results_df)}
for qt in query_types:
    summary[qt] = metrics_for(results_df[results_df['query_type'] == qt])

metrics_df = pd.DataFrame(summary).T
pd.set_option('display.float_format', lambda v: f"{v:.3f}")
print("RANKING METRICS (rows = query type, cols = metric)\n")
print(metrics_df.to_string())

overall_mrr = summary["OVERALL"]["MRR"]
print(f"\nHeadline: Hit@5 = {summary['OVERALL']['Hit@5']:.1%} | "
      f"MRR = {overall_mrr:.3f} | nDCG@10 = {summary['OVERALL']['nDCG@10']:.3f}")

## Secondary metrics — score-health diagnostics

The original four metrics, kept for monitoring. They describe how the cosine
scores *behave*; they do **not** establish that results are correct (the ranking
metrics above do that). Read them as supporting signals.

In [ ]:
# Diagnostic 1 - Average similarity score (score magnitude, NOT correctness)
print("DIAGNOSTIC 1 - Average similarity score")
for qt in query_types:
    subset = results_df[results_df['query_type'] == qt]
    avg    = subset['top_score'].mean()
    above  = (subset['top_score'] >= MIN_SCORE).sum()
    print(f"  {qt:8} | avg top score: {avg:.4f} | >= {MIN_SCORE}: {above}/{len(subset)}")
overall_avg = results_df['top_score'].mean()
print(f"\n  Overall avg top score: {overall_avg:.4f} "
      f"({'above' if overall_avg >= MIN_SCORE else 'below'} {MIN_SCORE} threshold)")

In [ ]:
# Diagnostic 2 - EN vs FR score gap (bilingual balance)
print("DIAGNOSTIC 2 - EN vs FR score gap")
en_avg = results_df[results_df['search_idx'] == 'en']['top_score'].mean()
fr_avg = results_df[results_df['search_idx'] == 'fr']['top_score'].mean()
gap    = abs(en_avg - fr_avg)
print(f"  EN-index avg: {en_avg:.4f} | FR-index avg: {fr_avg:.4f} | gap: {gap:.4f}")
band = ("EXCELLENT" if gap < 0.01 else "GOOD" if gap < 0.05
        else "ACCEPTABLE" if gap < 0.10 else "POOR - consider FR fine-tuning")
print(f"  Balance: {band}")

In [ ]:
# Diagnostic 3 - Cross-lingual consistency
# For the same source record, how much do the EN top-k and FR top-k overlap?
print("DIAGNOSTIC 3 - Cross-lingual consistency")
en_dict = {q['true_idx']: q for q in all_queries if q['query_type'] == 'en->en'}
fr_dict = {q['true_idx']: q for q in all_queries if q['query_type'] == 'fr->fr'}
common  = set(en_dict) & set(fr_dict)
print(f"  Matched {len(common)} record pairs")

consistency_scores = []
for idx in common:
    en_idx, _ = search(en_dict[idx]['query'], 'en', TOP_K)
    fr_idx, _ = search(fr_dict[idx]['query'], 'fr', TOP_K)
    consistency_scores.append(len(set(en_idx) & set(fr_idx)) / TOP_K)

avg_consistency = (np.mean(consistency_scores) * 100) if consistency_scores else 0.0
label = ("STRONG" if avg_consistency >= 60 else
         "MODERATE" if avg_consistency >= 40 else "WEAK")
print(f"  Avg top-{TOP_K} overlap: {avg_consistency:.1f}%  ->  {label}")

In [ ]:
# Diagnostic 4 - Score distribution (spread / reliability of top scores)
print("DIAGNOSTIC 4 - Score distribution")
for qt in query_types:
    s = results_df[results_df['query_type'] == qt]['top_score']
    print(f"  {qt:8} | min {s.min():.3f} | max {s.max():.3f} "
          f"| mean {s.mean():.3f} | std {s.std():.4f}")
print("  (lower std = more consistent confidence)")

## Visualization

Top row: the **ranking metrics** (the headline result). Bottom row: the score
diagnostics.

In [ ]:
colors = ['#2196F3', '#4CAF50', '#FF9800', '#E91E63']
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("Bilingual Semantic Search - Evaluation", fontsize=14, fontweight='bold')

# (0,0) Hit Rate@k curves per query type  -- PRIMARY
for qt, c in zip(query_types, colors):
    ranks = results_df[results_df['query_type'] == qt]['gold_rank'].tolist()
    axes[0,0].plot(K_VALUES, [hit_rate_at_k(ranks, k) for k in K_VALUES],
                   marker='o', color=c, label=qt)
axes[0,0].set_title("Hit Rate@k by query type (higher = better)")
axes[0,0].set_xlabel("k"); axes[0,0].set_ylabel("Hit Rate")
axes[0,0].set_ylim(0, 1.02); axes[0,0].set_xticks(K_VALUES)
axes[0,0].grid(alpha=0.3); axes[0,0].legend(fontsize=8)

# (0,1) MRR & nDCG@10 per query type  -- PRIMARY
x = np.arange(len(query_types)); w = 0.38
mrr_vals  = [mrr(results_df[results_df['query_type']==qt]['gold_rank'].tolist())
             for qt in query_types]
ndcg_vals = [ndcg_at_k(results_df[results_df['query_type']==qt]['gold_rank'].tolist(), 10)
             for qt in query_types]
axes[0,1].bar(x - w/2, mrr_vals,  w, label='MRR',     color='#3F51B5')
axes[0,1].bar(x + w/2, ndcg_vals, w, label='nDCG@10', color='#009688')
axes[0,1].set_title("MRR and nDCG@10 by query type")
axes[0,1].set_xticks(x); axes[0,1].set_xticklabels(query_types)
axes[0,1].set_ylim(0, 1.02); axes[0,1].legend(fontsize=8)
for xi, v in zip(x - w/2, mrr_vals):  axes[0,1].text(xi, v+0.02, f"{v:.2f}", ha='center', fontsize=8)
for xi, v in zip(x + w/2, ndcg_vals): axes[0,1].text(xi, v+0.02, f"{v:.2f}", ha='center', fontsize=8)

# (1,0) EN vs FR avg score  -- DIAGNOSTIC
axes[1,0].bar(['EN index', 'FR index'], [en_avg, fr_avg],
              color=['#2196F3', '#4CAF50'], width=0.4)
axes[1,0].set_title(f"Avg top score: EN vs FR (gap {gap:.4f})")
axes[1,0].set_ylabel("Avg top score"); axes[1,0].set_ylim(0.75, 0.95)
for i, v in enumerate([en_avg, fr_avg]):
    axes[1,0].text(i, v + 0.002, f"{v:.4f}", ha='center', fontweight='bold')

# (1,1) Score distribution  -- DIAGNOSTIC
for qt, c in zip(query_types, colors):
    s = results_df[results_df['query_type'] == qt]['top_score']
    axes[1,1].hist(s, bins=12, alpha=0.5, color=c, label=qt)
axes[1,1].set_title("Top-score distribution by query type")
axes[1,1].set_xlabel("Similarity score"); axes[1,1].set_ylabel("Count")
axes[1,1].legend(fontsize=8)

plt.tight_layout()
plt.savefig(data_folder + "evaluation_results.png", dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Persist per-query results and the metric summary
results_df.to_csv(data_folder + "evaluation_results.csv", index=False)
metrics_df.to_csv(data_folder + "evaluation_metrics_summary.csv")
print("Saved evaluation_results.csv, evaluation_metrics_summary.csv, "
      "evaluation_results.png")

## Optional — On-Topic Rate (arXiv:2410.21549)

The ranking metrics above need a known gold record per query. For *organic*
queries where there is no single right answer, the LinkedIn paper proposes
**On-Topic Rate (OTR@K)**: retrieve top-K, ask an LLM to judge each `(query, doc)`
pair as on-topic (1) or not (0), and report the share of relevant results.

Below is a provider-agnostic scaffold — plug in any LLM `judge` function. It is
left un-run (set `RUN_OTR = True` to use it).

In [ ]:
RUN_OTR = False  # set True once you wire up `judge(query, doc_text) -> 0/1`

def judge(query, doc_text):
    """Return 1 if doc_text is on-topic for query, else 0.
    Replace this stub with a real LLM call (OpenAI/Anthropic/local).
    The paper uses a short instruction + few-shot guidance and parses a 0/1."""
    raise NotImplementedError("Wire up an LLM judge before running OTR.")

def otr_at_k(queries, k=5, text_col='title_en'):
    rows = []
    for q in queries:
        idx, _ = search(q['query'], q['search_idx'], k)
        docs   = dataframe.loc[idx, text_col].tolist()
        judged = [judge(q['query'], d) for d in docs]
        rows.append(np.mean(judged))
    return float(np.mean(rows))

if RUN_OTR:
    score = otr_at_k(all_queries, k=5)
    print(f"OTR@5 = {score:.1%}")
else:
    print("OTR skipped (RUN_OTR = False).")